# NumPy for Edge AI — Real-World Pipelines & Patterns

**Module 2.2 | Edge AI Engineering Bootcamp**

This notebook brings together everything from the Basics, Intermediate, and Advanced notebooks and applies it to **real Edge AI scenarios** — the kind of code you'll actually write when deploying models on a Raspberry Pi or similar edge device.

## What You'll Learn

1. Complete camera-to-model inference pipeline
2. Image preprocessing: resize, normalize, standardize
3. Post-processing: confidence thresholds, NMS basics, label mapping
4. Batch processing for throughput optimization
5. Memory-conscious patterns for constrained devices
6. Common Edge AI bugs and how to avoid them
7. Putting it all together: a mini object detection pipeline

In [1]:
import numpy as np
import time
import sys

---
## 1. The Edge AI Data Pipeline Overview

Every Edge AI vision pipeline follows roughly the same flow:

```
Camera → Raw Bytes → NumPy Array → Preprocess → Model Input → Inference → Postprocess → Output
         (uint8)    (uint8)      (float32)    (float32)    (raw)      (filtered)   (result)
```

Each step involves NumPy operations. Understanding the dtype and shape at each stage is critical:

| Stage | Typical dtype | Typical shape | Notes |
|---|---|---|---|
| Camera raw bytes | bytes/uint8 | (H×W×C,) flat | Raw buffer from hardware |
| NumPy array | uint8 | (H, W, C) | Reshaped from buffer |
| Preprocessed | float32 | (H, W, C) or (C, H, W) | Normalized, standardized |
| Model input | float32 | (1, H, W, C) or (1, C, H, W) | Batched for inference |
| Model output | float32 | varies | Raw predictions |
| Postprocessed | uint8/float32 | varies | Thresholded, labeled |

In [2]:
# --- Simulating the full pipeline step by step ---

# Step 1: Camera produces raw bytes (simulated)
CAMERA_H, CAMERA_W, CAMERA_C = 480, 640, 3
raw_frame = np.random.randint(0, 256, 
    CAMERA_H * CAMERA_W * CAMERA_C, dtype=np.uint8).tobytes()

print("=== Step 1: Camera Raw Bytes ===")
print(f"  Size: {len(raw_frame):,} bytes")
print(f"  Type: {type(raw_frame)}")

# Step 2: Convert to NumPy array (zero-copy with frombuffer)
frame = np.frombuffer(raw_frame, dtype=np.uint8).reshape((CAMERA_H, CAMERA_W, CAMERA_C))
print(f"\n=== Step 2: NumPy Array ===")
print(f"  Shape: {frame.shape}")
print(f"  Dtype: {frame.dtype}")
print(f"  Memory: {frame.nbytes / 1024 / 1024:.2f} MB")
print(f"  Writable: {frame.flags.writeable}")

=== Step 1: Camera Raw Bytes ===
  Size: 921,600 bytes
  Type: <class 'bytes'>

=== Step 2: NumPy Array ===
  Shape: (480, 640, 3)
  Dtype: uint8
  Memory: 0.88 MB
  Writable: False


In [3]:
# Step 3: Preprocessing — normalize to [0, 1] float32
# This is where most Edge AI bugs happen (dtype issues, overflow, shape mismatches)

# Make a writable copy first (frombuffer gives read-only array)
frame_writable = frame.copy()

# Convert to float32 and normalize to [0.0, 1.0]
frame_normalized = frame_writable.astype(np.float32) / 255.0

print("=== Step 3: Preprocessing ===")
print(f"  Shape: {frame_normalized.shape}")
print(f"  Dtype: {frame_normalized.dtype}")
print(f"  Range: [{frame_normalized.min():.3f}, {frame_normalized.max():.3f}]")
print(f"  Memory: {frame_normalized.nbytes / 1024 / 1024:.2f} MB (4x more than uint8!)")

# Alternative: standardize with ImageNet mean/std (common for pretrained models)
imagenet_mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
imagenet_std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# Broadcasting: (480, 640, 3) - (3,) → (480, 640, 3)
frame_standardized = (frame_normalized - imagenet_mean) / imagenet_std
print(f"\n  After ImageNet standardization:")
print(f"  Range: [{frame_standardized.min():.3f}, {frame_standardized.max():.3f}]")
print(f"  Per-channel mean: {frame_standardized.mean(axis=(0,1))}")

=== Step 3: Preprocessing ===
  Shape: (480, 640, 3)
  Dtype: float32
  Range: [0.000, 1.000]
  Memory: 3.52 MB (4x more than uint8!)

  After ImageNet standardization:
  Range: [-2.118, 2.640]
  Per-channel mean: [0.07054619 0.19450317 0.41906637]


In [4]:
# Step 4: Resize (simulated — in real code you'd use OpenCV or PIL)
# Many models expect 224x224 input (like MobileNet, ResNet, etc.)

MODEL_INPUT_SIZE = 224

# Simple resize by cropping and padding (not high-quality, but demonstrates the concept)
# In production, use cv2.resize() or PIL Image.resize()
h, w = frame_normalized.shape[:2]

# Center crop to square (take the smaller dimension)
crop_size = min(h, w)
y_start = (h - crop_size) // 2
x_start = (w - crop_size) // 2
cropped = frame_normalized[y_start:y_start + crop_size, x_start:x_start + crop_size]
print(f"=== Step 4a: Center Crop ===")
print(f"  Cropped shape: {cropped.shape}")

# Simple downsampling by strided slicing (take every Nth pixel)
# For real resizing, use interpolation (cv2.resize, etc.)
scale = crop_size / MODEL_INPUT_SIZE
step = int(np.ceil(scale))
resized = cropped[::step, ::step][:MODEL_INPUT_SIZE, :MODEL_INPUT_SIZE]

# If the result is smaller than needed, pad with zeros
if resized.shape[0] < MODEL_INPUT_SIZE or resized.shape[1] < MODEL_INPUT_SIZE:
    padded = np.zeros((MODEL_INPUT_SIZE, MODEL_INPUT_SIZE, 3), dtype=np.float32)
    padded[:resized.shape[0], :resized.shape[1]] = resized
    resized = padded

print(f"\n=== Step 4b: Resized ===")
print(f"  Resized shape: {resized.shape}")
print(f"  Dtype: {resized.dtype}")
print(f"  Range: [{resized.min():.3f}, {resized.max():.3f}]")

=== Step 4a: Center Crop ===
  Cropped shape: (480, 480, 3)

=== Step 4b: Resized ===
  Resized shape: (224, 224, 3)
  Dtype: float32
  Range: [0.000, 1.000]


In [5]:
# Step 5: Add batch dimension and prepare for model input

# Most models expect a batch dimension: (batch, height, width, channels)
# Some models (PyTorch) expect (batch, channels, height, width) — check your model!

# TensorFlow/Keras style: (1, 224, 224, 3) — NHWC
input_tf = np.expand_dims(resized, axis=0)  # or: resized[np.newaxis, ...]
print("=== Step 5: Batch Input ===")
print(f"  TensorFlow style (NHWC): {input_tf.shape}")  # (1, 224, 224, 3)

# PyTorch style: (1, 3, 224, 224) — NCHW (channels first)
input_pt = np.transpose(input_tf, (0, 3, 1, 2))  # Move channels to position 1
print(f"  PyTorch style (NCHW):    {input_pt.shape}")  # (1, 3, 224, 224)

# ⚠️ After transpose, the array is NOT contiguous — fix it before passing to C libraries
print(f"  Contiguous after transpose? {input_pt.flags['C_CONTIGUOUS']}")  # False
input_pt_safe = np.ascontiguousarray(input_pt)
print(f"  After ascontiguousarray?   {input_pt_safe.flags['C_CONTIGUOUS']}")  # True

# Final check: dtype must be float32 for most models
print(f"\n  Final dtype: {input_pt_safe.dtype}")
print(f"  Final shape: {input_pt_safe.shape}")
print(f"  Ready for model inference!")

=== Step 5: Batch Input ===
  TensorFlow style (NHWC): (1, 224, 224, 3)
  PyTorch style (NCHW):    (1, 3, 224, 224)
  Contiguous after transpose? False
  After ascontiguousarray?   True

  Final dtype: float32
  Final shape: (1, 3, 224, 224)
  Ready for model inference!


---
## 2. Simulated Model Inference & Post-Processing

In a real project, you'd pass the preprocessed input to a model (TensorFlow Lite, ONNX Runtime, etc.). Here we simulate the model output and focus on **post-processing with NumPy**.

In [6]:
# --- Simulated classification model output ---

# A classification model typically outputs a vector of logits or probabilities
# Simulate 5-class classification (e.g., person, car, dog, cat, background)
class_names = ['person', 'car', 'dog', 'cat', 'background']
num_classes = len(class_names)

# Simulate model output (raw logits — unnormalized scores)
logits = np.array([2.1, -0.5, 5.8, 1.2, -3.0], dtype=np.float32)
print("=== Classification Post-Processing ===")
print(f"  Raw logits: {logits}")

# Convert logits to probabilities using softmax
# softmax(x) = exp(x) / sum(exp(x))
exp_logits = np.exp(logits - logits.max())  # subtract max for numerical stability
probabilities = exp_logits / exp_logits.sum()
print(f"  Probabilities: {probabilities}")
print(f"  Sum (should be 1.0): {probabilities.sum():.6f}")

# Get the predicted class
predicted_idx = probabilities.argmax()
predicted_class = class_names[predicted_idx]
confidence = probabilities[predicted_idx]
print(f"\n  Predicted: {predicted_class} (confidence: {confidence:.2%})")

# Apply a confidence threshold (common in Edge AI — don't act on low-confidence predictions)
CONFIDENCE_THRESHOLD = 0.7
if confidence >= CONFIDENCE_THRESHOLD:
    print(f"  → ACCEPTED (confidence >= {CONFIDENCE_THRESHOLD})")
else:
    print(f"  → REJECTED (confidence < {CONFIDENCE_THRESHOLD})")

# Show top-3 predictions
top3_idx = probabilities.argsort()[::-1][:3]
print(f"\n  Top-3 predictions:")
for idx in top3_idx:
    print(f"    {class_names[idx]}: {probabilities[idx]:.2%}")

=== Classification Post-Processing ===
  Raw logits: [ 2.1 -0.5  5.8  1.2 -3. ]
  Probabilities: [2.3846852e-02 1.7711913e-03 9.6454120e-01 9.6954061e-03 1.4538823e-04]
  Sum (should be 1.0): 1.000000

  Predicted: dog (confidence: 96.45%)
  → ACCEPTED (confidence >= 0.7)

  Top-3 predictions:
    dog: 96.45%
    person: 2.38%
    cat: 0.97%


In [7]:
# --- Simulated object detection model output ---

# Object detection models output multiple detections with:
# - bounding box (x, y, width, height or x1, y1, x2, y2)
# - class label
# - confidence score

# Simulate 10 raw detections: [class_id, confidence, x1, y1, x2, y2]
detections = np.array([
    [0, 0.92, 100, 150, 200, 300],  # person, high confidence
    [1, 0.85, 300, 200, 450, 350],  # car, high confidence
    [0, 0.78, 110, 160, 210, 310],  # person (overlapping with first)
    [2, 0.65, 50, 400, 120, 460],   # dog, medium confidence
    [0, 0.45, 500, 100, 580, 200],  # person, low confidence
    [3, 0.30, 350, 250, 400, 320],  # cat, low confidence
    [1, 0.88, 310, 210, 460, 360],  # car (overlapping with second)
    [2, 0.72, 55, 405, 125, 465],   # dog (overlapping with fourth)
    [0, 0.15, 600, 50, 630, 120],   # person, very low confidence
    [4, 0.90, 0, 0, 640, 480],      # background, high confidence
], dtype=np.float32)

print("=== Object Detection Post-Processing ===")
print(f"  Raw detections: {detections.shape[0]}")

# Step 1: Filter by confidence threshold
CONF_THRESHOLD = 0.5
confident_mask = detections[:, 1] >= CONF_THRESHOLD
confident_detections = detections[confident_mask]
print(f"  After confidence filter (>= {CONF_THRESHOLD}): {confident_detections.shape[0]}")

# Step 2: Sort by confidence (highest first)
sort_idx = confident_detections[:, 1].argsort()[::-1]
sorted_detections = confident_detections[sort_idx]
print(f"\n  Sorted by confidence:")
for det in sorted_detections:
    cls_id, conf, x1, y1, x2, y2 = det
    cls_name = class_names[int(cls_id)]
    print(f"    {cls_name:12s} conf={conf:.2f}  bbox=({x1:.0f},{y1:.0f})-({x2:.0f},{y2:.0f})")

=== Object Detection Post-Processing ===
  Raw detections: 10
  After confidence filter (>= 0.5): 7

  Sorted by confidence:
    person       conf=0.92  bbox=(100,150)-(200,300)
    background   conf=0.90  bbox=(0,0)-(640,480)
    car          conf=0.88  bbox=(310,210)-(460,360)
    car          conf=0.85  bbox=(300,200)-(450,350)
    person       conf=0.78  bbox=(110,160)-(210,310)
    dog          conf=0.72  bbox=(55,405)-(125,465)
    dog          conf=0.65  bbox=(50,400)-(120,460)


In [8]:
# --- Non-Maximum Suppression (NMS) — simplified version ---

# NMS removes overlapping detections, keeping only the most confident one
# This is a core post-processing step in every object detection pipeline

def compute_iou(box1, box2):
    """Compute Intersection over Union (IoU) between two bounding boxes.
    Boxes are in [x1, y1, x2, y2] format."""
    # Intersection coordinates
    ix1 = max(box1[0], box2[0])
    iy1 = max(box1[1], box2[1])
    ix2 = min(box1[2], box2[2])
    iy2 = min(box1[3], box2[3])
    
    # Intersection area
    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)
    intersection = iw * ih
    
    # Union area
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0

def simple_nms(detections, iou_threshold=0.5):
    """Simple Non-Maximum Suppression.
    detections: array of [class_id, confidence, x1, y1, x2, y2]
    Returns filtered detections with overlapping boxes removed."""
    if len(detections) == 0:
        return detections
    
    # Sort by confidence (highest first)
    idx = detections[:, 1].argsort()[::-1]
    detections = detections[idx]
    
    keep = []
    suppress = set()
    
    for i in range(len(detections)):
        if i in suppress:
            continue
        keep.append(i)
        
        # Compare with all remaining detections
        for j in range(i + 1, len(detections)):
            if j in suppress:
                continue
            
            # Only suppress if same class
            if detections[i, 0] == detections[j, 0]:
                iou = compute_iou(detections[i, 2:6], detections[j, 2:6])
                if iou > iou_threshold:
                    suppress.add(j)
    
    return detections[keep]

# Apply NMS
final_detections = simple_nms(sorted_detections, iou_threshold=0.5)
print(f"After NMS: {final_detections.shape[0]} detections (was {sorted_detections.shape[0]})")
print("\nFinal detections:")
for det in final_detections:
    cls_id, conf, x1, y1, x2, y2 = det
    cls_name = class_names[int(cls_id)]
    print(f"  {cls_name:12s} conf={conf:.2f}  bbox=({x1:.0f},{y1:.0f})-({x2:.0f},{y2:.0f})")

After NMS: 4 detections (was 7)

Final detections:
  person       conf=0.92  bbox=(100,150)-(200,300)
  background   conf=0.90  bbox=(0,0)-(640,480)
  car          conf=0.88  bbox=(310,210)-(460,360)
  dog          conf=0.72  bbox=(55,405)-(125,465)


---
## 3. Batch Processing for Throughput

On edge devices, processing multiple frames at once (batching) can improve throughput by utilizing the hardware more efficiently. However, memory is limited, so batch size must be chosen carefully.

In [9]:
# --- Building a batch of images ---

# Simulate capturing 4 frames
frames = []
for i in range(4):
    frame = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)
    frames.append(frame)

# Build a batch by stacking along a new axis
batch = np.stack(frames, axis=0)
print(f"Batch shape: {batch.shape}")  # (4, 480, 640, 3)
print(f"Batch dtype: {batch.dtype}")
print(f"Batch memory: {batch.nbytes / 1024 / 1024:.2f} MB")

# Preprocess the entire batch at once (vectorised!)
batch_normalized = batch.astype(np.float32) / 255.0
print(f"\nNormalized batch shape: {batch_normalized.shape}")
print(f"Normalized batch memory: {batch_normalized.nbytes / 1024 / 1024:.2f} MB")

# Per-frame statistics (axis=0 aggregates across the batch)
batch_means = batch_normalized.mean(axis=(1, 2))  # mean per frame per channel
print(f"\nPer-frame, per-channel means: shape={batch_means.shape}")  # (4, 3)
for i in range(4):
    print(f"  Frame {i}: R={batch_means[i, 0]:.3f}, G={batch_means[i, 1]:.3f}, B={batch_means[i, 2]:.3f}")

# Memory budget calculation for a Raspberry Pi (1 GB available RAM)
AVAILABLE_RAM_MB = 1024
frame_size_mb = (480 * 640 * 3 * 4) / 1024 / 1024  # float32
max_batch = int(AVAILABLE_RAM_MB * 0.3 / frame_size_mb)  # Use at most 30% of RAM
print(f"\n=== Memory Budget ===")
print(f"  One float32 frame: {frame_size_mb:.2f} MB")
print(f"  Available RAM (30% budget): {AVAILABLE_RAM_MB * 0.3:.0f} MB")
print(f"  Max batch size: {max_batch} frames")

Batch shape: (4, 480, 640, 3)
Batch dtype: uint8
Batch memory: 3.52 MB

Normalized batch shape: (4, 480, 640, 3)
Normalized batch memory: 14.06 MB

Per-frame, per-channel means: shape=(4, 3)
  Frame 0: R=0.501, G=0.499, B=0.501
  Frame 1: R=0.501, G=0.501, B=0.500
  Frame 2: R=0.500, G=0.500, B=0.500
  Frame 3: R=0.500, G=0.500, B=0.500

=== Memory Budget ===
  One float32 frame: 3.52 MB
  Available RAM (30% budget): 307 MB
  Max batch size: 87 frames


In [10]:
# --- Batch vs single-frame processing benchmark ---

# Simulate preprocessing 8 frames
frames = [np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8) for _ in range(8)]

# Method 1: Process one at a time
start = time.time()
results_single = []
for frame in frames:
    processed = frame.astype(np.float32) / 255.0
    results_single.append(processed)
results_single = np.stack(results_single)
t_single = time.time() - start

# Method 2: Batch process all at once
start = time.time()
batch = np.stack(frames, axis=0)
results_batch = batch.astype(np.float32) / 255.0
t_batch = time.time() - start

print(f"Process 8 frames individually: {t_single:.4f}s")
print(f"Process 8 frames as batch:     {t_batch:.4f}s")
print(f"Speedup: {t_single / t_batch:.2f}x")
print(f"Results match: {np.allclose(results_single, results_batch)}")
print(f"\n→ Batching is more efficient due to better cache utilization and fewer Python overhead calls")

Process 8 frames individually: 0.0014s
Process 8 frames as batch:     0.0012s
Speedup: 1.15x
Results match: True

→ Batching is more efficient due to better cache utilization and fewer Python overhead calls


---
## 4. Memory-Conscious Patterns for Edge Devices

On a Raspberry Pi with 1-4 GB RAM, every byte counts. Here are patterns to minimize memory usage.

In [11]:
# --- Memory-saving pattern 1: In-place operations ---

img = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8).astype(np.float32)
print(f"Before processing: {img.nbytes / 1024 / 1024:.2f} MB")

# BAD: each operation creates a new array (3 copies in memory at some point)
# result = (img / 255.0 - 0.5) * 2.0  # Creates intermediate arrays

# GOOD: in-place operations modify the existing array
img /= 255.0    # in-place division
img -= 0.5      # in-place subtraction
img *= 2.0      # in-place multiplication
print(f"After in-place processing: {img.nbytes / 1024 / 1024:.2f} MB (same array, no extra copies)")
print(f"Range: [{img.min():.3f}, {img.max():.3f}]")

# ⚠️ In-place ops modify the original array — only use when you don't need the original anymore!

Before processing: 3.52 MB
After in-place processing: 3.52 MB (same array, no extra copies)
Range: [-1.000, 1.000]


In [12]:
# --- Memory-saving pattern 2: Use the smallest sufficient dtype ---

# For model weights that don't need float64 precision, use float32
weights_f64 = np.random.randn(1000, 1000)  # default is float64
weights_f32 = weights_f64.astype(np.float32)

print(f"Weights as float64: {weights_f64.nbytes / 1024 / 1024:.2f} MB")
print(f"Weights as float32: {weights_f32.nbytes / 1024 / 1024:.2f} MB")
print(f"Savings: {weights_f64.nbytes / weights_f32.nbytes:.1f}x less memory")

# For quantized models (like TensorFlow Lite int8), use int8 for weights
# This can reduce model size by 4x compared to float32
weights_int8 = np.clip(weights_f32 * 127, -128, 127).astype(np.int8)
print(f"\nWeights as int8 (quantized): {weights_int8.nbytes / 1024 / 1024:.2f} MB")
print(f"Total savings vs float64: {weights_f64.nbytes / weights_int8.nbytes:.1f}x less memory")

# For intermediate computations, use float32 instead of float64
# Most edge AI models are designed for float32 precision

Weights as float64: 7.63 MB
Weights as float32: 3.81 MB
Savings: 2.0x less memory

Weights as int8 (quantized): 0.95 MB
Total savings vs float64: 8.0x less memory


In [13]:
# --- Memory-saving pattern 3: Process frames in a streaming fashion ---

# BAD: load all frames into memory at once
# all_frames = [capture_frame() for _ in range(1000)]  # Could OOM on a Pi!

# GOOD: process one frame at a time, keep only results
def simulate_streaming_pipeline(num_frames=100):
    """Process frames one at a time, keeping only the results."""
    results = []
    for i in range(num_frames):
        # Capture (simulated)
        frame = np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8)
        
        # Preprocess
        processed = frame.astype(np.float32) / 255.0
        
        # Simulate inference (just compute mean as a placeholder)
        result = processed.mean(axis=(0, 1))  # shape (3,) — tiny!
        results.append(result)
        
        # Frame goes out of scope here — memory is freed
    
    return np.array(results)

streaming_results = simulate_streaming_pipeline(100)
print(f"Processed 100 frames")
print(f"Results shape: {streaming_results.shape}")  # (100, 3) — tiny!
print(f"Results memory: {streaming_results.nbytes} bytes")
print(f"\n→ Only 100×3×4 = 1200 bytes kept, not 100×224×224×3×4 = 59 MB!")

Processed 100 frames
Results shape: (100, 3)
Results memory: 1200 bytes

→ Only 100×3×4 = 1200 bytes kept, not 100×224×224×3×4 = 59 MB!


In [14]:
# --- Memory-saving pattern 4: Reuse pre-allocated buffers ---

# For real-time video processing, pre-allocate buffers once and reuse them

class FrameProcessor:
    """Process video frames with pre-allocated buffers to avoid repeated allocation."""
    def __init__(self, height, width, channels=3):
        self.h, self.w, self.c = height, width, channels
        # Pre-allocate buffers (allocated once, reused for every frame)
        self._float_buffer = np.empty((height, width, channels), dtype=np.float32)
        self._temp_buffer = np.empty((height, width, channels), dtype=np.float32)
    
    def process(self, frame_uint8):
        """Process a single frame using pre-allocated buffers."""
        # Convert to float32 in-place into the pre-allocated buffer
        np.copyto(self._float_buffer, frame_uint8.astype(np.float32))
        
        # Normalize in-place
        self._float_buffer /= 255.0
        
        # Compute statistics
        mean = self._float_buffer.mean(axis=(0, 1))
        return mean

processor = FrameProcessor(480, 640, 3)
print(f"Pre-allocated buffers: {processor._float_buffer.nbytes / 1024 / 1024:.2f} MB (allocated once)")

# Process 100 frames — no new allocations after init!
start = time.time()
for _ in range(100):
    frame = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)
    result = processor.process(frame)
elapsed = time.time() - start
print(f"Processed 100 frames in {elapsed:.4f}s ({100/elapsed:.1f} FPS)")
print(f"Last result: R={result[0]:.3f}, G={result[1]:.3f}, B={result[2]:.3f}")

Pre-allocated buffers: 3.52 MB (allocated once)
Processed 100 frames in 0.3257s (307.0 FPS)
Last result: R=0.500, G=0.500, B=0.500


---
## 5. Common Edge AI Bugs and How to Avoid Them

In [15]:
# --- Bug 1: Height/Width swap ---

# BUG: Swapping height and width when cropping
img = np.zeros((480, 640, 3), dtype=np.uint8)  # 480 high, 640 wide

# WRONG: thinking (width, height) — this crops the wrong region
# wrong_crop = img[200:500, 100:300]  # This takes rows 200-499, cols 100-299
# But the student MEANT to crop x:200-500, y:100-300

# CORRECT: always use (y1:y2, x1:x2) = (height_start:height_end, width_start:width_end)
correct_crop = img[100:300, 200:500]  # rows 100-299 (height), cols 200-499 (width)
print(f"Correct crop shape: {correct_crop.shape}")  # (200, 300, 3) — 200 high, 300 wide

print("\n=== Rule ===")
print("  img[y1:y2, x1:x2] — ALWAYS height (rows) first, then width (columns)")
print("  Shape is always (height, width, channels)")

Correct crop shape: (200, 300, 3)

=== Rule ===
  img[y1:y2, x1:x2] — ALWAYS height (rows) first, then width (columns)
  Shape is always (height, width, channels)


In [16]:
# --- Bug 2: uint8 overflow in image arithmetic ---

img = np.array([200, 210, 220], dtype=np.uint8)

# BUG: Direct addition wraps around
buggy = img + 60  # 200+60=260 → wraps to 4!
print(f"BUG: {img} + 60 = {buggy} (uint8 wraps around!)")

# FIX: Upcast, add, clip, convert back
fixed = np.clip(img.astype(np.int16) + 60, 0, 255).astype(np.uint8)
print(f"FIX: {img} + 60 = {fixed} (clipped to 255)")

print("\n=== Rule ===")
print("  Always upcast (int16/float32) before arithmetic on uint8 images")
print("  Use np.clip() to prevent overflow, then convert back to uint8")

BUG: [200 210 220] + 60 = [ 4 14 24] (uint8 wraps around!)
FIX: [200 210 220] + 60 = [255 255 255] (clipped to 255)

=== Rule ===
  Always upcast (int16/float32) before arithmetic on uint8 images
  Use np.clip() to prevent overflow, then convert back to uint8


In [17]:
# --- Bug 3: Modifying a view instead of a copy ---

img = np.random.randint(0, 256, (4, 4, 3), dtype=np.uint8)
original_pixel = img[1, 1].copy()

# BUG: Modifying a slice modifies the original (slice is a VIEW)
region = img[1:3, 1:3]  # This is a view!
region[:] = 0           # This also modifies img!
print(f"BUG: Original pixel [1,1] was {original_pixel}, now is {img[1, 1]}")

# FIX: Use .copy() if you want to modify independently
img2 = np.random.randint(0, 256, (4, 4, 3), dtype=np.uint8)
original_pixel2 = img2[1, 1].copy()
region2 = img2[1:3, 1:3].copy()  # Explicit copy
region2[:] = 0
print(f"FIX: Original pixel [1,1] was {original_pixel2}, still is {img2[1, 1]}")

print("\n=== Rule ===")
print("  Slicing returns a VIEW — modifications affect the original")
print("  Use .copy() when you need an independent array")

BUG: Original pixel [1,1] was [234  89 162], now is [0 0 0]
FIX: Original pixel [1,1] was [ 53 221  98], still is [ 53 221  98]

=== Rule ===
  Slicing returns a VIEW — modifications affect the original
  Use .copy() when you need an independent array


In [18]:
# --- Bug 4: Non-contiguous array passed to external library ---

img = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)

# BUG: Transposing then passing to OpenCV/TFLite without fixing contiguity
transposed = img.transpose(1, 0, 2)  # Non-contiguous!
print(f"BUG: Transposed C_CONTIGUOUS = {transposed.flags['C_CONTIGUOUS']}")
print("  → Passing to OpenCV/TFLite may cause errors or wrong results")

# FIX: Use ascontiguousarray before passing to external libraries
fixed = np.ascontiguousarray(transposed)
print(f"\nFIX: After ascontiguousarray, C_CONTIGUOUS = {fixed.flags['C_CONTIGUOUS']}")
print("  → Safe to pass to OpenCV/TFLite")

print("\n=== Rule ===")
print("  After transpose/flip/step-slice, check .flags['C_CONTIGUOUS']")
print("  Use np.ascontiguousarray() before passing to external C libraries")

BUG: Transposed C_CONTIGUOUS = False
  → Passing to OpenCV/TFLite may cause errors or wrong results

FIX: After ascontiguousarray, C_CONTIGUOUS = True
  → Safe to pass to OpenCV/TFLite

=== Rule ===
  After transpose/flip/step-slice, check .flags['C_CONTIGUOUS']
  Use np.ascontiguousarray() before passing to external C libraries


In [19]:
# --- Bug 5: Wrong channel order (RGB vs BGR) ---

# OpenCV uses BGR by default, but most AI models expect RGB
# This is a common source of silently wrong predictions

# Simulate an image captured by OpenCV (BGR order)
img_bgr = np.array([[[255, 0, 0]]], dtype=np.uint8)  # Blue=255, Green=0, Red=0
print(f"OpenCV BGR image: {img_bgr[0, 0]} → Blue=255, Green=0, Red=0")

# Convert BGR to RGB (required for most AI models)
img_rgb = img_bgr[:, :, ::-1]  # Reverse channel order
print(f"After BGR→RGB:    {img_rgb[0, 0]} → Red=255, Green=0, Blue=0")

# ⚠️ After channel reversal, the array is NOT contiguous!
print(f"\nC_CONTIGUOUS after channel flip: {img_rgb.flags['C_CONTIGUOUS']}")
# Fix it:
img_rgb_safe = np.ascontiguousarray(img_rgb)
print(f"After ascontiguousarray:          {img_rgb_safe.flags['C_CONTIGUOUS']}")

print("\n=== Rule ===")
print("  OpenCV gives BGR — convert to RGB before feeding to AI models")
print("  Use img[:, :, ::-1] to reverse channels, then ascontiguousarray()")

OpenCV BGR image: [255   0   0] → Blue=255, Green=0, Red=0
After BGR→RGB:    [  0   0 255] → Red=255, Green=0, Blue=0

C_CONTIGUOUS after channel flip: False
After ascontiguousarray:          True

=== Rule ===
  OpenCV gives BGR — convert to RGB before feeding to AI models
  Use img[:, :, ::-1] to reverse channels, then ascontiguousarray()


---
## 6. Putting It All Together: Mini Object Detection Pipeline

Here's a complete (simulated) pipeline that ties together everything from all four notebooks.

In [20]:
# --- Complete Edge AI Pipeline (Simulated) ---

class EdgeAIPipeline:
    """A complete (simulated) Edge AI object detection pipeline.
    
    In a real project, the model inference step would call TensorFlow Lite,
    ONNX Runtime, or similar. Here we simulate it with random outputs.
    """
    
    def __init__(self, model_input_size=224, num_classes=5, conf_threshold=0.5, iou_threshold=0.4):
        self.input_size = model_input_size
        self.num_classes = num_classes
        self.conf_threshold = conf_threshold
        self.iou_threshold = iou_threshold
        self.class_names = ['person', 'car', 'dog', 'cat', 'background']
        
        # ImageNet normalization constants
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        
        # Pre-allocate buffers for efficiency
        self._input_buffer = np.empty((1, model_input_size, model_input_size, 3), dtype=np.float32)
    
    def preprocess(self, frame_uint8):
        """Convert raw camera frame to model input.
        
        Steps:
        1. Convert BGR to RGB (if from OpenCV)
        2. Convert to float32 and normalize to [0, 1]
        3. Standardize with ImageNet mean/std
        4. Resize to model input size (simulated)
        5. Add batch dimension
        6. Ensure contiguous memory layout
        """
        # Step 1: BGR → RGB (channel reversal)
        frame_rgb = frame_uint8[:, :, ::-1]
        
        # Step 2: uint8 → float32, normalize to [0, 1]
        frame_f32 = frame_rgb.astype(np.float32) / 255.0
        
        # Step 3: Standardize (broadcasting: (H,W,3) - (3,) → (H,W,3))
        frame_std = (frame_f32 - self.mean) / self.std
        
        # Step 4: Center crop + resize (simplified — use cv2.resize in production)
        h, w = frame_std.shape[:2]
        crop_size = min(h, w)
        y_start = (h - crop_size) // 2
        x_start = (w - crop_size) // 2
        cropped = frame_std[y_start:y_start + crop_size, x_start:x_start + crop_size]
        
        # Simple resize by strided sampling
        step = max(1, crop_size // self.input_size)
        resized = cropped[::step, ::step][:self.input_size, :self.input_size]
        
        # Pad if needed
        if resized.shape[0] < self.input_size or resized.shape[1] < self.input_size:
            padded = np.zeros((self.input_size, self.input_size, 3), dtype=np.float32)
            padded[:resized.shape[0], :resized.shape[1]] = resized
            resized = padded
        
        # Step 5: Add batch dimension: (H, W, C) → (1, H, W, C)
        batched = resized[np.newaxis, ...]
        
        # Step 6: Ensure contiguous (safe for external libraries)
        batched = np.ascontiguousarray(batched)
        
        return batched
    
    def inference(self, model_input):
        """Simulate model inference (replace with real model call in production)."""
        # Simulate detection output: [class_id, confidence, x1, y1, x2, y2] per detection
        num_detections = np.random.randint(3, 8)
        detections = np.zeros((num_detections, 6), dtype=np.float32)
        
        for i in range(num_detections):
            detections[i, 0] = np.random.randint(0, self.num_classes)  # class
            detections[i, 1] = np.random.uniform(0.3, 0.95)            # confidence
            detections[i, 2] = np.random.uniform(0, 400)               # x1
            detections[i, 3] = np.random.uniform(0, 300)               # y1
            detections[i, 4] = detections[i, 2] + np.random.uniform(50, 200)  # x2
            detections[i, 5] = detections[i, 3] + np.random.uniform(50, 200)  # y2
        
        return detections
    
    def postprocess(self, raw_detections, original_shape):
        """Post-process raw detections: filter, NMS, format output."""
        if len(raw_detections) == 0:
            return []
        
        # Filter by confidence threshold
        mask = raw_detections[:, 1] >= self.conf_threshold
        filtered = raw_detections[mask]
        
        if len(filtered) == 0:
            return []
        
        # Sort by confidence (highest first)
        idx = filtered[:, 1].argsort()[::-1]
        filtered = filtered[idx]
        
        # Non-Maximum Suppression
        keep = []
        suppress = set()
        for i in range(len(filtered)):
            if i in suppress:
                continue
            keep.append(i)
            for j in range(i + 1, len(filtered)):
                if j in suppress:
                    continue
                if filtered[i, 0] == filtered[j, 0]:  # Same class
                    iou = self._compute_iou(filtered[i, 2:6], filtered[j, 2:6])
                    if iou > self.iou_threshold:
                        suppress.add(j)
        
        final = filtered[keep]
        
        # Format output
        results = []
        for det in final:
            cls_id, conf, x1, y1, x2, y2 = det
            results.append({
                'class': self.class_names[int(cls_id)],
                'confidence': float(conf),
                'bbox': (int(x1), int(y1), int(x2), int(y2))
            })
        return results
    
    @staticmethod
    def _compute_iou(box1, box2):
        """Compute IoU between two boxes [x1, y1, x2, y2]."""
        ix1 = max(box1[0], box2[0])
        iy1 = max(box1[1], box2[1])
        ix2 = min(box1[2], box2[2])
        iy2 = min(box1[3], box2[3])
        iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
        intersection = iw * ih
        area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
        area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
        union = area1 + area2 - intersection
        return intersection / union if union > 0 else 0
    
    def run(self, frame_uint8):
        """Run the complete pipeline on a single frame."""
        original_shape = frame_uint8.shape
        model_input = self.preprocess(frame_uint8)
        raw_detections = self.inference(model_input)
        results = self.postprocess(raw_detections, original_shape)
        return results

print("EdgeAIPipeline class defined!")

EdgeAIPipeline class defined!


In [21]:
# --- Run the complete pipeline ---

pipeline = EdgeAIPipeline(model_input_size=224, conf_threshold=0.5)

# Simulate a camera frame (480x640x3, BGR, uint8 — as OpenCV would provide)
frame = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)

# Run the pipeline
start = time.time()
detections = pipeline.run(frame)
elapsed = time.time() - start

print(f"=== Pipeline Results ===")
print(f"Processing time: {elapsed * 1000:.1f} ms")
print(f"FPS equivalent: {1 / elapsed:.1f}")
print(f"Detections found: {len(detections)}")
print()
for i, det in enumerate(detections):
    print(f"  Detection {i+1}:")
    print(f"    Class:      {det['class']}")
    print(f"    Confidence: {det['confidence']:.2%}")
    print(f"    Bounding box: {det['bbox']}")

=== Pipeline Results ===
Processing time: 4.3 ms
FPS equivalent: 232.1
Detections found: 3

  Detection 1:
    Class:      car
    Confidence: 88.23%
    Bounding box: (345, 254, 496, 335)
  Detection 2:
    Class:      car
    Confidence: 64.32%
    Bounding box: (120, 267, 276, 398)
  Detection 3:
    Class:      dog
    Confidence: 53.09%
    Bounding box: (92, 259, 277, 434)


In [22]:
# --- Run multiple frames and measure performance ---

pipeline = EdgeAIPipeline(model_input_size=224, conf_threshold=0.5)

num_frames = 50
times = []
total_detections = 0

for i in range(num_frames):
    frame = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)
    
    start = time.time()
    detections = pipeline.run(frame)
    elapsed = time.time() - start
    
    times.append(elapsed)
    total_detections += len(detections)

times = np.array(times)
print(f"=== Performance Summary ({num_frames} frames) ===")
print(f"  Average time per frame: {times.mean() * 1000:.1f} ms")
print(f"  Std deviation:          {times.std() * 1000:.1f} ms")
print(f"  Min time:               {times.min() * 1000:.1f} ms")
print(f"  Max time:               {times.max() * 1000:.1f} ms")
print(f"  Average FPS:            {1 / times.mean():.1f}")
print(f"  Total detections:       {total_detections}")
print(f"  Avg detections/frame:   {total_detections / num_frames:.1f}")

# On a real Raspberry Pi, the preprocessing part would be similar,
# but the inference time would dominate (depends on model complexity).
# The NumPy preprocessing shown here is optimized and would run efficiently even on a Pi.

=== Performance Summary (50 frames) ===
  Average time per frame: 3.8 ms
  Std deviation:          0.3 ms
  Min time:               3.6 ms
  Max time:               4.8 ms
  Average FPS:            261.8
  Total detections:       175
  Avg detections/frame:   3.5


---
## 7. Complete Edge AI Checklist

Use this checklist when writing any Edge AI code with NumPy:

### Data Acquisition
- [ ] Use `np.frombuffer()` for zero-copy camera frame handling
- [ ] Check `frame.flags.writeable` — make a `.copy()` if you need to modify
- [ ] Verify shape is `(height, width, channels)` — NOT `(width, height, ...)`

### Preprocessing
- [ ] Convert BGR → RGB if using OpenCV (`frame[:, :, ::-1]`)
- [ ] Convert `uint8` → `float32` BEFORE arithmetic (`frame.astype(np.float32)`)
- [ ] Normalize to model's expected range (usually `[0, 1]` or `[-1, 1]`)
- [ ] Apply per-channel mean/std standardization if model requires it
- [ ] Resize to model's input dimensions
- [ ] Add batch dimension: `(H, W, C)` → `(1, H, W, C)`
- [ ] For PyTorch models: transpose to NCHW format
- [ ] Call `np.ascontiguousarray()` after any transpose/channel flip

### Performance
- [ ] **No Python loops over pixels** — use vectorised NumPy operations
- [ ] Use in-place operations (`*=`, `+=`, `/=`) to save memory
- [ ] Pre-allocate buffers for real-time processing
- [ ] Use `float32` (not `float64`) to save 50% memory
- [ ] Process frames in a streaming fashion (don't hold all in memory)

### Post-Processing
- [ ] Apply confidence threshold to filter low-confidence detections
- [ ] Apply NMS to remove overlapping detections
- [ ] Map class indices to human-readable labels

### Safety
- [ ] Always upcast `uint8` before arithmetic to prevent wraparound
- [ ] Use `np.clip()` to prevent overflow before converting back to `uint8`
- [ ] Use `.copy()` when you need to modify a slice independently
- [ ] Check `C_CONTIGUOUS` before passing arrays to external libraries

---
## Quick Recap — All Four Notebooks

### Basics (`numpy_basics.ipynb`)
1. NumPy arrays store same-type data in one contiguous memory block — this is why they're fast
2. Image shape is always `(height, width, channels)` — rows before columns
3. Slicing gives a **view**, not a copy — use `.copy()` for independent data
4. `dtype` determines memory usage: `uint8` = 1 byte, `float32` = 4 bytes
5. Strides enable instant zero-copy operations like flips and transposes

### Intermediate (`numpy_intermediate.ipynb`)
1. Broadcasting compares shapes right-to-left — matching if equal or one is `1`
2. `uint8` silently wraps around on overflow — always upcast before risky math
3. Safe arithmetic: `np.clip(img.astype(np.int16) + val, 0, 255).astype(np.uint8)`
4. Aggregations with `axis` give per-channel statistics instantly
5. Boolean masking enables thresholding without loops

### Advanced (`numpy_advanced.ipynb`)
1. `np.frombuffer()` — zero-copy camera frame handling (read-only by default)
2. `np.ascontiguousarray()` — fixes memory layout after transpose/flip
3. Vectorised operations are ~100x faster than Python loops (NEON SIMD)
4. Pre-allocate arrays and use in-place ops to minimize memory allocations
5. `np.einsum()` for complex matrix operations including batched matmul

### Edge AI (`numpy_edgeai.ipynb` — this notebook)
1. The pipeline: Camera → uint8 → float32 → normalize → standardize → batch → model → postprocess
2. Batch processing improves throughput but watch memory limits
3. Stream frames one at a time on memory-constrained devices
4. Pre-allocate and reuse buffers for real-time processing
5. Always check: dtype, shape, contiguity, and channel order before model inference

## Final Self-Check

1. Walk through the complete pipeline: what dtype and shape does your data have at each stage?
2. Why is it dangerous to add a scalar to a `uint8` image? What's the safe pattern?
3. Your model expects `(1, 3, 224, 224)` float32 input. Your camera gives `(480, 640, 3)` uint8 BGR. Write the preprocessing code.
4. You're processing 30 FPS video on a Pi with 1 GB RAM. What memory-saving patterns would you use?
5. After calling `img.transpose(1, 0, 2)`, what two things must you check before passing to TensorFlow Lite?